# Eredivisie Bayesian Finishing — Scouting Analysis

**Empirical Bayes Beta-Binomial model applied to Eredivisie shot data (2022–23 to 2024–25)**

This notebook loads the pre-computed Bayesian finishing estimates from `bayesian_finishing.csv`, derives all analytical metrics, produces visual summaries of finishing quality across the league, and exports a structured Excel scouting workbook.

---
**Key metric: `eb_mean`** — the Bayesian posterior mean conversion rate for each player, shrunk toward the league prior. More reliable than raw goals/shots for players with small samples.

**How shrinkage works:** a player with 5 shots is pulled heavily toward the league mean (~12–13%), while a player with 200 shots is barely adjusted. `reliability = 1 − shrinkage` tells you how much to trust the number.

---
*Place `bayesian_finishing.csv` in the same directory as this notebook, then run all cells top to bottom.*

---
## Section 1 — Setup

Install dependencies, import libraries, and configure file paths.

In [ ]:
!pip install pandas numpy openpyxl matplotlib seaborn scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side, GradientFill
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

import warnings
warnings.filterwarnings('ignore')

print(f"pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}")

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
CSV_PATH    = "bayesian_finishing.csv"   # place file alongside this notebook
OUTPUT_PATH = "eredivisie_scouting.xlsx" # Excel workbook output

# ── Colour palette ────────────────────────────────────────────────────────────
NAVY   = "#0D1B2A"
TEAL   = "#1B7A78"
GOLD   = "#E8C33A"
GREEN  = "#1DB954"
RED    = "#E05252"
ORANGE = "#F09B2A"
WHITE  = "#FFFFFF"

# Matplotlib global style
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': NAVY,
    'axes.facecolor':   '#111E2B',
    'axes.edgecolor':   '#2A3E52',
    'axes.labelcolor':  '#CCCCCC',
    'xtick.color':      '#AAAAAA',
    'ytick.color':      '#AAAAAA',
    'text.color':       '#EEEEEE',
    'grid.color':       '#1E2E3E',
    'grid.alpha':       0.6,
    'font.family':      'DejaVu Sans',
})

print("Config ready.")

---
## Section 2 — Load & Explore Data

Read the CSV produced by the Empirical Bayes model and inspect its structure. Each row represents one player across all three Eredivisie seasons (2022–23, 2023–24, 2024–25).

**Expected columns:**
| Column | Meaning |
|---|---|
| `player_name` | Player display name |
| `n` | Total non-penalty shots |
| `goals` | Total goals |
| `xg_sum` | Sum of pre-shot xG values |
| `raw_rate` | Naïve goals/shots |
| `eb_mean` | Bayesian posterior mean conversion rate |
| `eb_ci_lo / eb_ci_hi` | 90% credible interval bounds |
| `eb_vs_xg` | Finishing skill above/below xG expectation |
| `shrinkage` | How much pulled toward the prior (1 = all prior, 0 = all data) |
| `eb_alpha_post / eb_beta_post` | Posterior Beta shape parameters |

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}  ({df.shape[0]} players, {df.shape[1]} columns)")
print("\nColumn dtypes:")
print(df.dtypes)

In [ ]:
df.head(10)

In [ ]:
key_cols = ['n', 'goals', 'xg_sum', 'raw_rate', 'eb_mean', 'eb_vs_xg', 'shrinkage']
df[key_cols].describe().round(4)

---
## Section 3 — Derived Metrics

We compute additional columns that power the analysis:

| Derived Column | Formula | Interpretation |
|---|---|---|
| `goals_above_xg` | `goals − xg_sum` | Raw overperformance vs expected |
| `goals_above_exp_pct` | `goals_above_xg / goals` (clipped at 1) | Proportion of goals above xG |
| `ci_width` | `eb_ci_hi − eb_ci_lo` | Width of the 90% credible interval — smaller = more certain |
| `reliability` | `1 − shrinkage` | 1.0 = fully data-driven; 0.0 = all prior |
| `xg_per_shot` | `xg_sum / n` | Average shot quality — shot selection metric |

We also compute the **league mean conversion rate** from the shared EB prior (alpha, beta parameters common to all players), and derive **percentile ranks** and **tier labels**.

### Tier Thresholds

| Tier | EB Mean | Reliability | Notes |
|---|---|---|---|
| Elite | ≥ 15.5% | ≥ 55% | Primary scouting targets |
| Strong | ≥ 13.0% | ≥ 40% | Reliable above-average finishers |
| Average | ≥ 11.0% | — | League standard |
| Below Avg | ≥ 8.5% | — | Underperforming — context needed |
| Weak | < 8.5% | — | Poor finishing |
| Insufficient data | < 10 shots | — | Cannot assess reliably |

In [ ]:
# ── Core derived columns ──────────────────────────────────────────────────────
df["goals_above_xg"]    = df["goals"] - df["xg_sum"]
df["goals_above_exp_pct"] = df["goals_above_xg"] / df["goals"].clip(lower=1)
df["ci_width"]          = df["eb_ci_hi"] - df["eb_ci_lo"]
df["reliability"]       = 1 - df["shrinkage"]   # 1 = fully data-driven
df["xg_per_shot"]       = df["xg_sum"] / df["n"]
df["finishing_edge"]    = df["eb_vs_xg"]          # alias for clarity

print("Derived columns added:")
print(df[["goals_above_xg", "goals_above_exp_pct", "ci_width",
          "reliability", "xg_per_shot"]].describe().round(4))

In [ ]:
# ── League mean from EB prior ─────────────────────────────────────────────────
# Recover the prior alpha/beta by subtracting the observed data from the posteriors
df["alpha_prior"] = df["eb_alpha_post"] - df["goals"]
df["beta_prior"]  = df["eb_beta_post"]  - (df["n"] - df["goals"])

# The prior mean is alpha_prior / (alpha_prior + beta_prior) — same for all players
# We take the mean across rows as a robust estimate
league_mean = (df["alpha_prior"] / (df["alpha_prior"] + df["beta_prior"])).mean()

print(f"League EB prior mean: {league_mean*100:.3f}%")
print(f"(A player with zero shots would be assigned {league_mean*100:.2f}% conversion probability)")

In [ ]:
# ── Percentile ranks (100 = best in league) ───────────────────────────────────
for col, ascending in [
    ("eb_mean",     False),
    ("eb_vs_xg",    False),
    ("reliability", False),
    ("raw_rate",    False),
    ("n",           False),
]:
    df[f"pct_{col}"] = df[col].rank(pct=True, ascending=ascending) * 100

print("Percentile rank columns created:")
print([c for c in df.columns if c.startswith("pct_")])

In [ ]:
# ── Tier classification ───────────────────────────────────────────────────────
TIER_ORDER = {"Elite": 0, "Strong": 1, "Average": 2,
              "Below Avg": 3, "Weak": 4, "Insufficient data": 5}

TIER_COLOR_HEX = {
    "Elite":             GREEN,
    "Strong":            "#5BA85A",
    "Average":           "#A8A83A",
    "Below Avg":         "#E07A3A",
    "Weak":              RED,
    "Insufficient data": "#888888",
}

def assign_tier(row):
    if row["n"] < 10:                                                 return "Insufficient data"
    if row["eb_mean"] >= 0.155 and row["reliability"] >= 0.55:       return "Elite"
    if row["eb_mean"] >= 0.130 and row["reliability"] >= 0.40:       return "Strong"
    if row["eb_mean"] >= 0.110:                                       return "Average"
    if row["eb_mean"] >= 0.085:                                       return "Below Avg"
    return "Weak"

df["tier"]      = df.apply(assign_tier, axis=1)
df["tier_rank"] = df["tier"].map(TIER_ORDER)

print("Tier distribution:")
print(df["tier"].value_counts().reindex(TIER_ORDER.keys()))

In [ ]:
# ── Working subsets ───────────────────────────────────────────────────────────
df_q30 = df[df["n"] >= 30].copy()   # high-confidence estimates
df_q10 = df[df["n"] >= 10].copy()   # reliable enough for tier assignment
df_all = df.copy()

# Common sort orders used throughout
elite_sort   = df_q10.sort_values(["tier_rank", "eb_mean"], ascending=[True, False])
finedge_sort = df_q10.sort_values("eb_vs_xg", ascending=False)
volume_sort  = df_q30.sort_values("eb_mean",  ascending=False)
hidden_sort  = df[(df["n"] >= 5) & (df["n"] < 30)].sort_values("eb_mean", ascending=False)

print(f"Full dataset : {len(df):,} players")
print(f"≥10 shots    : {len(df_q10):,} players")
print(f"≥30 shots    : {len(df_q30):,} players")
print(f"Hidden gems  : {len(hidden_sort):,} players (5–29 shots)")

---
## Section 4 — League Overview

### The EB Mean and Tier Framework

The **EB Mean** (`eb_mean`) is the primary finishing quality metric in this analysis. It represents the Bayesian posterior mean conversion rate: how often we estimate a player would score from a shot, after accounting for sample size via shrinkage toward the league prior.

**Why not just use raw conversion rate?**  
A player with 3 goals from 10 shots has a raw rate of 30%, which is almost certainly noise rather than signal. The Bayesian model shrinks this toward the league mean (~12–13%), producing a more stable and predictive estimate.

The six tiers provide a quick scouting classification:
- **Elite (≥15.5% + ≥55% reliability):** These players demonstrably convert at above-league rates with sufficient data. Primary transfer targets.
- **Strong (≥13.0% + ≥40% reliability):** Reliable above-average finishers.
- **Average (≥11.0%):** Converting close to the league mean — value comes from other attributes.
- **Below Avg (≥8.5%)** and **Weak (<8.5%):** Finishing is a concern.
- **Insufficient data (<10 shots):** The model cannot distinguish signal from noise.

In [ ]:
# ── Tier distribution bar chart ───────────────────────────────────────────────
tier_counts = df_q10["tier"].value_counts().reindex(
    [k for k in TIER_ORDER if k != "Insufficient data"]
).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

colors = [TIER_COLOR_HEX[t] for t in tier_counts.index]
bars   = ax.bar(tier_counts.index, tier_counts.values, color=colors,
                edgecolor='#2A3E52', linewidth=0.8, width=0.65)

for bar, val in zip(bars, tier_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            str(val), ha='center', va='bottom', fontsize=11,
            fontweight='bold', color=WHITE)

ax.set_title("Tier Distribution — Eredivisie Finishers (min 10 shots)",
             fontsize=14, fontweight='bold', color=GOLD, pad=14)
ax.set_xlabel("Tier", fontsize=11, color='#CCCCCC')
ax.set_ylabel("Number of Players", fontsize=11, color='#CCCCCC')
ax.tick_params(axis='both', colors='#AAAAAA')
ax.grid(axis='y', alpha=0.3, color='#2A3E52')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#2A3E52')
ax.spines['bottom'].set_color('#2A3E52')

plt.tight_layout()
plt.show()

In [ ]:
# ── KPI Summary ───────────────────────────────────────────────────────────────
top_player     = df_q10.loc[df_q10['eb_mean'].idxmax(), 'player_name']
top_eb         = df_q10['eb_mean'].max()
elite_count    = (df_q10['tier'] == 'Elite').sum()
best_edge_name = df_q10.loc[df_q10['eb_vs_xg'].idxmax(), 'player_name']
best_edge_val  = df_q10['eb_vs_xg'].max()

print("═" * 55)
print("  EREDIVISIE BAYESIAN FINISHING — KPI SUMMARY")
print("═" * 55)
print(f"  Total players analysed  : {len(df):,}")
print(f"  Qualified (≥10 shots)   : {len(df_q10):,}")
print(f"  Elite finishers         : {elite_count}")
print(f"  League EB prior mean    : {league_mean*100:.2f}%")
print(f"  Top EB mean             : {top_eb*100:.2f}%  ({top_player})")
print(f"  Best finishing edge     : +{best_edge_val*100:.2f}pp  ({best_edge_name})")
print(f"  Hidden gems (5–29 shots): {len(hidden_sort)}")
print("═" * 55)

---
## Section 5 — Elite Finishers

Players classified as **Elite** meet both criteria:
1. `eb_mean ≥ 15.5%` — converting meaningfully above league average
2. `reliability ≥ 55%` — we have enough shots that the data outweighs the prior

These are the **primary scouting targets**: the model is confident in their ability, not just reporting a lucky streak.

The chart below shows the full EB credible interval for each player — wider bars mean more uncertainty.

In [ ]:
elite_display = df_q10[df_q10['tier'] == 'Elite'].sort_values('eb_mean', ascending=False)
print(f"Elite finishers: {len(elite_display)}\n")
elite_display[['player_name', 'n', 'goals', 'raw_rate', 'eb_mean',
               'eb_ci_lo', 'eb_ci_hi', 'eb_vs_xg', 'reliability']].round(4)

In [ ]:
# ── Top 20 by eb_mean — horizontal bar with CI error bars ─────────────────────
top20 = df_q10.nlargest(20, 'eb_mean').sort_values('eb_mean', ascending=True)

fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

y      = range(len(top20))
colors = [TIER_COLOR_HEX.get(t, '#888888') for t in top20['tier']]

xerr_lo = top20['eb_mean'] - top20['eb_ci_lo']
xerr_hi = top20['eb_ci_hi'] - top20['eb_mean']

ax.barh(y, top20['eb_mean'] * 100, color=colors, alpha=0.85,
        edgecolor='#2A3E52', linewidth=0.6, height=0.7)
ax.errorbar(top20['eb_mean'] * 100, y,
            xerr=[xerr_lo * 100, xerr_hi * 100],
            fmt='none', ecolor=WHITE, elinewidth=1.2,
            capsize=3, capthick=1.2, alpha=0.7)

# League mean reference line
ax.axvline(league_mean * 100, color=GOLD, linestyle='--',
           linewidth=1.2, alpha=0.7, label=f'League mean ({league_mean*100:.1f}%)')

ax.set_yticks(list(y))
ax.set_yticklabels(top20['player_name'], fontsize=9, color=WHITE)
ax.set_xlabel('EB Posterior Mean Conversion Rate (%)', fontsize=10, color='#CCCCCC')
ax.set_title('Top 20 Finishers — EB Mean with 90% Credible Intervals',
             fontsize=13, fontweight='bold', color=GOLD, pad=12)

# Tier legend
patches = [mpatches.Patch(color=TIER_COLOR_HEX[t], label=t)
           for t in ['Elite', 'Strong', 'Average'] if t in top20['tier'].values]
patches.append(mpatches.Patch(color=GOLD, label=f'League mean ({league_mean*100:.1f}%)'))
ax.legend(handles=patches, loc='lower right', fontsize=8,
          facecolor='#1A2B3C', edgecolor='#2A3E52')

ax.grid(axis='x', alpha=0.3, color='#2A3E52')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#2A3E52')
ax.spines['bottom'].set_color('#2A3E52')

plt.tight_layout()
plt.show()

---
## Section 6 — Finishing Edge (EB vs xG)

**`eb_vs_xg = eb_mean − xg_per_shot`**

This metric separates two sources of conversion rate:
1. **Shot quality** (`xg_per_shot`): the player gets into good positions → high xG chances
2. **Pure finishing skill** (`eb_vs_xg`): the player converts *more* (or less) than even those chances would predict

A large positive `eb_vs_xg` is a strong signal of genuine finishing quality — the player is not just getting lucky with high-xG chances, they're also executing under pressure.

A large negative `eb_vs_xg` may indicate poor technique, bad luck, or a role where positional quality doesn't translate to goals.

In [ ]:
# ── Top 20 overperformers ─────────────────────────────────────────────────────
top_overperf = df_q10.nlargest(20, 'eb_vs_xg').sort_values('eb_vs_xg', ascending=True)

fig, ax = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

y = range(len(top_overperf))
ax.barh(y, top_overperf['eb_vs_xg'] * 100, color=TEAL, alpha=0.85,
        edgecolor='#2A3E52', linewidth=0.6, height=0.7)
ax.axvline(0, color='#AAAAAA', linewidth=1.0, linestyle='-')

ax.set_yticks(list(y))
ax.set_yticklabels(top_overperf['player_name'], fontsize=9, color=WHITE)
ax.set_xlabel('Finishing Edge: EB Mean − xG/Shot (percentage points)', fontsize=10, color='#CCCCCC')
ax.set_title('Top 20 Overperformers — EB Mean above xG Rate',
             fontsize=13, fontweight='bold', color=GOLD, pad=12)

ax.grid(axis='x', alpha=0.3, color='#2A3E52')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#2A3E52')
ax.spines['bottom'].set_color('#2A3E52')
plt.tight_layout()
plt.show()

In [ ]:
# ── Bottom 10 underperformers ─────────────────────────────────────────────────
bot_under = df_q10.nsmallest(10, 'eb_vs_xg').sort_values('eb_vs_xg', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

y = range(len(bot_under))
ax.barh(y, bot_under['eb_vs_xg'] * 100, color=RED, alpha=0.85,
        edgecolor='#2A3E52', linewidth=0.6, height=0.7)
ax.axvline(0, color='#AAAAAA', linewidth=1.0, linestyle='-')

ax.set_yticks(list(y))
ax.set_yticklabels(bot_under['player_name'], fontsize=9, color=WHITE)
ax.set_xlabel('Finishing Edge: EB Mean − xG/Shot (percentage points)', fontsize=10, color='#CCCCCC')
ax.set_title('Bottom 10 Underperformers — EB Mean below xG Rate',
             fontsize=13, fontweight='bold', color=RED, pad=12)

ax.grid(axis='x', alpha=0.3, color='#2A3E52')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#2A3E52')
ax.spines['bottom'].set_color('#2A3E52')
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: xg_per_shot vs eb_mean, sized by n, coloured by eb_vs_xg ─────────
plot_df = df_q10.copy()

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

sc = ax.scatter(
    plot_df['xg_per_shot'] * 100,
    plot_df['eb_mean']     * 100,
    c     = plot_df['eb_vs_xg'] * 100,
    s     = np.clip(plot_df['n'] / 3, 15, 200),
    cmap  = 'RdYlGn',
    alpha = 0.75,
    edgecolors = 'none',
    vmin  = -6,
    vmax  = 6,
)

cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('EB vs xG (pp)', color='#CCCCCC', fontsize=9)
cbar.ax.yaxis.set_tick_params(color='#AAAAAA')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#AAAAAA')

# Reference lines
ax.axhline(league_mean * 100, color=GOLD, linestyle='--', linewidth=1.0, alpha=0.6,
           label=f'League EB mean ({league_mean*100:.1f}%)')
xg_mean = plot_df['xg_per_shot'].mean() * 100
ax.axvline(xg_mean, color='#AAAAAA', linestyle='--', linewidth=1.0, alpha=0.5,
           label=f'Mean xG/shot ({xg_mean:.1f}%)')

ax.set_xlabel('xG per Shot — Shot Quality (%)', fontsize=11, color='#CCCCCC')
ax.set_ylabel('EB Mean Conversion Rate (%)',     fontsize=11, color='#CCCCCC')
ax.set_title('Shot Quality vs Finishing Quality\n(bubble size = shots; colour = EB above/below xG)',
             fontsize=13, fontweight='bold', color=GOLD, pad=12)
ax.legend(fontsize=8, facecolor='#1A2B3C', edgecolor='#2A3E52')

ax.grid(alpha=0.3, color='#2A3E52')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#2A3E52')
ax.spines['bottom'].set_color('#2A3E52')

plt.tight_layout()
plt.show()

---
## Section 7 — Shot Selection vs Finishing Quality

This scatter plot places players in a 2×2 matrix:

| Quadrant | xG/Shot | EB Mean | Type |
|---|---|---|---|
| Top-right | High | High | **Elite all-round** — great positions, great finishing |
| Top-left | Low | High | **Poor positions, good finish** — clinical despite limited chances |
| Bottom-right | High | Low | **Good positions, poor finish** — gets into chances but wastes them |
| Bottom-left | Low | Low | **Weak** — neither quality positions nor strong finishing |

Minimum 30 shots ensures we only see reliable estimates here.

In [ ]:
# ── Quadrant scatter: shot selection vs finishing (min 30 shots) ──────────────
q_df = df_q30.copy()

xg_med  = q_df['xg_per_shot'].median() * 100
eb_med  = q_df['eb_mean'].median()     * 100

fig, ax = plt.subplots(figsize=(12, 9))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

colors_q = [TIER_COLOR_HEX.get(t, '#888888') for t in q_df['tier']]

ax.scatter(
    q_df['xg_per_shot'] * 100,
    q_df['eb_mean']     * 100,
    c        = colors_q,
    s        = np.clip(q_df['n'] / 2, 20, 250),
    alpha    = 0.80,
    edgecolors = 'none',
)

# Quadrant dividers
ax.axvline(xg_med,  color='#AAAAAA', linestyle='--', linewidth=1.0, alpha=0.5)
ax.axhline(eb_med,  color='#AAAAAA', linestyle='--', linewidth=1.0, alpha=0.5)

# Quadrant labels
xlim = ax.get_xlim(); ylim = ax.get_ylim()
pad_x = (xlim[1] - xlim[0]) * 0.03
pad_y = (ylim[1] - ylim[0]) * 0.03

ax.text(xlim[1] - pad_x, ylim[1] - pad_y, "Elite all-round",
        ha='right', va='top', fontsize=9, color=GREEN,  alpha=0.85, style='italic')
ax.text(xlim[0] + pad_x, ylim[1] - pad_y, "Poor positions,\ngood finish",
        ha='left',  va='top', fontsize=9, color=GOLD,   alpha=0.85, style='italic')
ax.text(xlim[1] - pad_x, ylim[0] + pad_y, "Good positions,\npoor finish",
        ha='right', va='bottom', fontsize=9, color=ORANGE, alpha=0.85, style='italic')
ax.text(xlim[0] + pad_x, ylim[0] + pad_y, "Weak",
        ha='left',  va='bottom', fontsize=9, color=RED,    alpha=0.85, style='italic')

# Annotate top 15 players by eb_mean
top15 = q_df.nlargest(15, 'eb_mean')
for _, row in top15.iterrows():
    ax.annotate(
        row['player_name'],
        xy     = (row['xg_per_shot'] * 100, row['eb_mean'] * 100),
        xytext = (5, 3),
        textcoords = 'offset points',
        fontsize = 7.5,
        color    = WHITE,
        alpha    = 0.9,
    )

ax.set_xlabel('xG per Shot — Shot Selection Quality (%)',
              fontsize=11, color='#CCCCCC')
ax.set_ylabel('EB Mean Conversion Rate (%)',
              fontsize=11, color='#CCCCCC')
ax.set_title('Shot Selection vs Finishing Quality (min 30 shots)\nTop 15 by EB Mean annotated',
             fontsize=13, fontweight='bold', color=GOLD, pad=12)

# Tier legend
patches = [mpatches.Patch(color=TIER_COLOR_HEX[t], label=t)
           for t in ['Elite', 'Strong', 'Average', 'Below Avg', 'Weak']
           if t in q_df['tier'].values]
ax.legend(handles=patches, loc='upper left', fontsize=8,
          facecolor='#1A2B3C', edgecolor='#2A3E52')

ax.grid(alpha=0.25, color='#2A3E52')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#2A3E52')
ax.spines['bottom'].set_color('#2A3E52')

plt.tight_layout()
plt.show()

---
## Section 8 — Hidden Gems

Players with only **5–29 shots** but an `eb_mean ≥ 12.5%` represent high-potential early leads. The Bayesian model still shrinks their estimates heavily toward the prior (high shrinkage), so these are **signals to investigate further**, not confirmed elite finishers.

The **verdict logic** for each player:
- `eb_mean ≥ 15.0%` → **Strong signal — scout immediately**
- `eb_mean ≥ 13.0% AND eb_vs_xg > +1pp` → **Promising — outperforms xG, needs more data**
- `eb_mean ≥ 12.0%` → **Interesting — follow for next 10+ shots**
- Otherwise → **Inconclusive — insufficient evidence**

The value of catching these players early: if their signal proves real at 50+ shots, the market will have already repriced them.

In [ ]:
# ── Hidden gems filter ────────────────────────────────────────────────────────
def hidden_verdict(row):
    if row['eb_mean'] >= 0.150:
        return 'Strong signal — scout immediately'
    if row['eb_mean'] >= 0.130 and row['eb_vs_xg'] > 0.010:
        return 'Promising — outperforms xG, needs more data'
    if row['eb_mean'] >= 0.120:
        return 'Interesting — follow for next 10+ shots'
    return 'Inconclusive — insufficient evidence'

gems_df = df[(df['n'] >= 5) & (df['n'] < 30) & (df['eb_mean'] >= 0.125)].copy()
gems_df = gems_df.sort_values('eb_mean', ascending=False)
gems_df['verdict'] = gems_df.apply(hidden_verdict, axis=1)

print(f"Hidden gems: {len(gems_df)} players\n")
gems_df[['player_name', 'n', 'goals', 'raw_rate', 'eb_mean',
         'eb_vs_xg', 'shrinkage', 'verdict']].head(20)

In [ ]:
# ── Bar chart: top 15 hidden gems ─────────────────────────────────────────────
gems_top15 = gems_df.head(15).sort_values('eb_mean', ascending=True)

verdict_colors = {
    'Strong signal — scout immediately':            GREEN,
    'Promising — outperforms xG, needs more data':  TEAL,
    'Interesting — follow for next 10+ shots':      GOLD,
    'Inconclusive — insufficient evidence':         '#888888',
}
bar_colors = [verdict_colors.get(v, '#888888') for v in gems_top15['verdict']]

fig, ax = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

y = range(len(gems_top15))
ax.barh(y, gems_top15['eb_mean'] * 100, color=bar_colors, alpha=0.85,
        edgecolor='#2A3E52', linewidth=0.6, height=0.7)
ax.axvline(league_mean * 100, color=GOLD, linestyle='--', linewidth=1.0,
           alpha=0.6, label=f'League mean ({league_mean*100:.1f}%)')

# Shot count labels
for i, (_, row) in enumerate(gems_top15.iterrows()):
    ax.text(row['eb_mean'] * 100 + 0.1, i,
            f" n={int(row['n'])}", va='center', fontsize=7.5, color='#CCCCCC')

ax.set_yticks(list(y))
ax.set_yticklabels(gems_top15['player_name'], fontsize=9, color=WHITE)
ax.set_xlabel('EB Mean Conversion Rate (%)', fontsize=10, color='#CCCCCC')
ax.set_title('Top 15 Hidden Gems — High EB Mean with 5–29 Shots\n(colour = scouting verdict)',
             fontsize=13, fontweight='bold', color=ORANGE, pad=12)

legend_handles = [mpatches.Patch(color=c, label=v)
                  for v, c in verdict_colors.items()]
ax.legend(handles=legend_handles, loc='lower right', fontsize=7.5,
          facecolor='#1A2B3C', edgecolor='#2A3E52')

ax.grid(axis='x', alpha=0.3, color='#2A3E52')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#2A3E52')
ax.spines['bottom'].set_color('#2A3E52')

plt.tight_layout()
plt.show()

---
## Section 9 — Key Insights Summary

The seven structured insights that also appear in the Excel workbook. Each insight targets a specific scouting question.

### Insight 1 — Elite Finishers

Players with `eb_mean ≥ 15.5%` **and** `reliability ≥ 55%`. These are statistically confirmed elite finishers — the data sample is large enough that the Bayesian shrinkage has little effect. Primary transfer targets for any attacking role.

In [ ]:
insight1 = df_q10[df_q10['tier'] == 'Elite'].sort_values('eb_mean', ascending=False)
print(f"Elite finishers: {len(insight1)}")
insight1[['player_name', 'n', 'goals', 'raw_rate', 'eb_mean', 'eb_vs_xg', 'reliability']]

### Insight 2 — Consistent Overperformers

Players with `eb_vs_xg > +2.5pp`, `min 30 shots`, and growing reliability. These are not just getting into good positions — they're converting *better* than even their high-xG chances would predict. Highly transferable skill.

In [ ]:
insight2 = (
    df_q30[(df_q30['eb_vs_xg'] > 0.025) & (df_q30['reliability'] > 0.25)]
    .sort_values('eb_vs_xg', ascending=False)
    .head(12)
)
print(f"Consistent overperformers: {len(insight2)}")
insight2[['player_name', 'n', 'goals', 'xg_sum', 'xg_per_shot', 'eb_mean', 'eb_vs_xg', 'reliability']]

### Insight 3 — Shot Selection Leaders

Players with `xg_per_shot > 16%` and `min 30 shots`. These players consistently get into the highest-quality scoring positions. Combined with strong finishing, this is the most dangerous profile.

In [ ]:
insight3 = (
    df_q30[df_q30['xg_per_shot'] > 0.16]
    .sort_values('xg_per_shot', ascending=False)
    .head(12)
)
print(f"Shot selection leaders: {len(insight3)}")
insight3[['player_name', 'n', 'goals', 'xg_per_shot', 'eb_mean', 'eb_vs_xg', 'reliability']]

### Insight 4 — Goals Above Expected

`goals_above_xg = goals − xg_sum`. The raw count of goals above what was expected from shot quality. Sorted by this metric (min 30 shots), these are the players who have added the most value through pure finishing across their Eredivisie careers in this dataset.

In [ ]:
insight4 = df_q30.nlargest(12, 'goals_above_xg')
print(f"Goals above expected leaders: {len(insight4)}")
insight4[['player_name', 'n', 'goals', 'xg_sum', 'goals_above_xg', 'eb_mean', 'eb_vs_xg']]

### Insight 5 — Buy-Low Candidates

Players with **high shot quality** (`xg_per_shot > 14%`) but **negative finishing edge** (`eb_vs_xg < −3pp`), min 30 shots. These are players whose market value may be suppressed by finishing underperformance, despite getting into excellent positions. Could improve with coaching or regress positively toward their xG.

In [ ]:
insight5 = (
    df_q30[(df_q30['xg_per_shot'] > 0.14) & (df_q30['eb_vs_xg'] < -0.03)]
    .sort_values('eb_vs_xg')
    .head(10)
)
print(f"Buy-low candidates: {len(insight5)}")
insight5[['player_name', 'n', 'goals', 'xg_per_shot', 'eb_mean', 'eb_vs_xg', 'shrinkage']]

### Insight 6 — Hidden Gems

Players with **5–29 shots** and `eb_mean ≥ 12.5%`. Small-sample leaders — high shrinkage, so treat these as scouting leads rather than confirmed signals. Catching these before the market does is the primary use case.

In [ ]:
insight6 = df[(df['n'] >= 5) & (df['n'] < 30) & (df['eb_mean'] >= 0.125)].sort_values('eb_mean', ascending=False).head(15)
insight6['verdict'] = insight6.apply(hidden_verdict, axis=1)
print(f"Hidden gems: {len(insight6)}")
insight6[['player_name', 'n', 'goals', 'raw_rate', 'eb_mean', 'eb_vs_xg', 'shrinkage', 'verdict']]

### Insight 7 — Volume + Quality

Players with `eb_mean > league_mean + 3pp` ranked by shot volume. These are the proven, high-usage finishers — the model has seen enough data that the estimates are highly reliable. Consistency and volume together are the most bankable combination.

In [ ]:
insight7 = df[df['eb_mean'] > league_mean + 0.03].nlargest(8, 'n')
print(f"Volume + quality leaders: {len(insight7)}")
insight7[['player_name', 'n', 'goals', 'xg_sum', 'goals_above_xg', 'eb_mean', 'reliability']]

---
## Section 10 — Export

Export the full analysis to both CSV files (for quick use) and a structured Excel workbook with multiple sheets:

1. **Scout Overview** — tier distribution summary and KPIs
2. **Elite Targets** — players sorted by tier and EB mean
3. **Finishing Edge** — sorted by `eb_vs_xg`
4. **Hidden Gems** — small-sample leaders
5. **Full Rankings** — all players sorted by EB mean

The workbook uses the NAVY/TEAL/GOLD colour scheme and conditional formatting on key columns.

In [ ]:
# ── CSV exports ───────────────────────────────────────────────────────────────
df.sort_values('eb_mean', ascending=False).to_csv('full_rankings.csv', index=False)
df_q10[df_q10['tier'] == 'Elite'].sort_values('eb_mean', ascending=False).to_csv('elite_finishers.csv', index=False)
gems_df.to_csv('hidden_gems.csv', index=False)
finedge_sort.to_csv('finishing_edge.csv', index=False)
print("CSV exports written:")
print("  full_rankings.csv")
print("  elite_finishers.csv")
print("  hidden_gems.csv")
print("  finishing_edge.csv")

In [ ]:
# ── openpyxl helper functions ─────────────────────────────────────────────────
# Colours as openpyxl hex strings (no #)
_NAVY     = "0D1B2A"
_TEAL     = "1B7A78"
_GOLD     = "E8C33A"
_WHITE    = "FFFFFF"
_LIGHT_BG = "F4F7FA"
_GREY     = "C5CDD6"
_DARK_TXT = "0D1B2A"

TIER_COLOR_XL = {
    "Elite":             "1DB954",
    "Strong":            "5BA85A",
    "Average":           "A8A83A",
    "Below Avg":         "E07A3A",
    "Weak":              "E05252",
    "Insufficient data": "AAAAAA",
}

def xl_fill(hex_str):
    return PatternFill("solid", fgColor=hex_str)

def xl_font(bold=False, size=10, color=_DARK_TXT, italic=False):
    return Font(bold=bold, size=size, color=color, italic=italic, name="Calibri")

def xl_border():
    s = Side(style="thin", color=_GREY)
    return Border(left=s, right=s, top=s, bottom=s)

def xl_center():
    return Alignment(horizontal="center", vertical="center", wrap_text=False)

def xl_left(wrap=False):
    return Alignment(horizontal="left", vertical="center", wrap_text=wrap)

def xl_pct(v, d=1):
    if pd.isna(v): return ""
    return f"{v*100:.{d}f}%"

def xl_header_row(ws, row, headers, widths, bg=_TEAL, height=20):
    for ci, (h, w) in enumerate(zip(headers, widths), 1):
        ws.column_dimensions[get_column_letter(ci)].width = w
        c = ws.cell(row=row, column=ci, value=h)
        c.font      = xl_font(bold=True, color=_WHITE, size=9)
        c.fill      = xl_fill(bg)
        c.alignment = xl_center()
        c.border    = xl_border()
    ws.row_dimensions[row].height = height

def xl_title(ws, row, col1, col2, text, bg=_NAVY, size=14, height=32):
    ws.merge_cells(start_row=row, start_column=col1, end_row=row, end_column=col2)
    c = ws.cell(row=row, column=col1, value=text)
    c.font      = Font(bold=True, size=size, color=_WHITE, name="Calibri")
    c.fill      = xl_fill(bg)
    c.alignment = xl_center()
    ws.row_dimensions[row].height = height

print("Helper functions defined.")

In [ ]:
# ── Build Excel workbook ──────────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

# ──────────────────────────────────────────────────────────────────────────────
# Sheet 1: Scout Overview
# ──────────────────────────────────────────────────────────────────────────────
ws1 = wb.create_sheet("Scout Overview")
ws1.sheet_view.showGridLines = False

# Title
xl_title(ws1, 1, 1, 14, "EREDIVISIE SCOUTING REPORT — BAYESIAN FINISHING ANALYSIS", height=36)
ws1.merge_cells("A2:N2")
sub = ws1.cell(row=2, column=1,
               value=f"Empirical Bayes Beta-Binomial  |  {len(df)} Players  |  "
                     f"Seasons 2022-23 to 2024-25  |  League Mean: {league_mean*100:.2f}%")
sub.font      = Font(size=9, color=_WHITE, italic=True, name="Calibri")
sub.fill      = xl_fill(_TEAL)
sub.alignment = xl_center()
ws1.row_dimensions[2].height = 18

# Tier summary table
tier_groups_xl = (
    df_q10.groupby("tier")
    .agg(
        players  = ("player_name", "count"),
        eb_mean  = ("eb_mean", "mean"),
        n        = ("n", "mean"),
        goals    = ("goals", "mean"),
        edge     = ("eb_vs_xg", "mean"),
        ci_w     = ("ci_width", "mean"),
    )
    .reset_index()
)
tier_groups_xl["rank"] = tier_groups_xl["tier"].map(TIER_ORDER)
tier_groups_xl = tier_groups_xl.sort_values("rank")

# Add insufficient data row
insuf = df[df["tier"] == "Insufficient data"]
extra = pd.DataFrame([{
    "tier": "Insufficient data",
    "players": len(insuf),
    "eb_mean": insuf["eb_mean"].mean() if len(insuf) else 0,
    "n": insuf["n"].mean() if len(insuf) else 0,
    "goals": insuf["goals"].mean() if len(insuf) else 0,
    "edge": insuf["eb_vs_xg"].mean() if len(insuf) else 0,
    "ci_w": insuf["ci_width"].mean() if len(insuf) else 0,
    "rank": 5,
}])
tier_groups_xl = pd.concat([tier_groups_xl, extra], ignore_index=True)

tier_headers_xl = ["Tier", "Players", "Avg EB Mean", "Avg Shots",
                   "Avg Goals", "Avg Edge", "Avg CI Width"]
tier_widths_xl  = [16, 9, 13, 11, 11, 12, 13]
xl_header_row(ws1, 4, tier_headers_xl, tier_widths_xl, bg=_NAVY)

for ri, rd in enumerate(tier_groups_xl.itertuples(), 5):
    tc  = TIER_COLOR_XL.get(rd.tier, "AAAAAA")
    rbg = _LIGHT_BG if ri % 2 == 0 else _WHITE
    vals = [
        rd.tier,
        int(rd.players),
        f"{rd.eb_mean*100:.2f}%",
        f"{rd.n:.0f}",
        f"{rd.goals:.1f}",
        f"{rd.edge*100:+.2f}%",
        f"{rd.ci_w*100:.2f}pp",
    ]
    for ci, val in enumerate(vals, 1):
        c = ws1.cell(row=ri, column=ci, value=val)
        c.border    = xl_border()
        c.alignment = xl_center()
        if ci == 1:
            c.font = Font(bold=True, size=10, color=_WHITE, name="Calibri")
            c.fill = xl_fill(tc)
        else:
            c.font = xl_font(size=10)
            c.fill = xl_fill(rbg)
    ws1.row_dimensions[ri].height = 18

print("Sheet 1: Scout Overview — done")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Sheet 2: Elite Targets
# ──────────────────────────────────────────────────────────────────────────────
ws2 = wb.create_sheet("Elite Targets")
ws2.sheet_view.showGridLines = False

xl_title(ws2, 1, 1, 13, "ELITE FINISHING TARGETS — Sorted by Tier, then EB Mean")

headers2 = ["#", "Player", "Tier", "Shots", "Goals", "Raw Conv%",
            "xG/Shot", "EB Mean", "CI Low", "CI High",
            "EB vs xG", "Reliability", "Scouting Note"]
widths2  = [5, 22, 14, 7, 7, 10, 10, 10, 10, 10, 11, 11, 40]
xl_header_row(ws2, 2, headers2, widths2)

def scouting_note(row):
    notes = []
    if row["tier"] == "Elite":            notes.append("Primary target")
    if row["eb_vs_xg"] > 0.030:           notes.append(f"outperforms xG by {row['eb_vs_xg']*100:.1f}pp")
    elif row["eb_vs_xg"] < -0.030:        notes.append(f"underperforms xG by {abs(row['eb_vs_xg'])*100:.1f}pp")
    if row["reliability"] > 0.70:         notes.append("high reliability")
    elif row["reliability"] < 0.40:       notes.append("small sample — verify")
    if row["ci_width"] < 0.06:            notes.append("tight CI")
    return " | ".join(notes) if notes else "Meets league standard"

for ri, (_, row) in enumerate(elite_sort.iterrows(), 3):
    tc  = TIER_COLOR_XL.get(row["tier"], "AAAAAA")
    rbg = _LIGHT_BG if ri % 2 == 1 else _WHITE
    vals = [
        ri - 2, row["player_name"], row["tier"],
        int(row["n"]), int(row["goals"]),
        xl_pct(row["raw_rate"]), xl_pct(row["xg_per_shot"]),
        xl_pct(row["eb_mean"]),  xl_pct(row["eb_ci_lo"]),
        xl_pct(row["eb_ci_hi"]), f"{row['eb_vs_xg']*100:+.2f}%",
        xl_pct(row["reliability"]), scouting_note(row),
    ]
    for ci, val in enumerate(vals, 1):
        c = ws2.cell(row=ri, column=ci, value=val)
        c.border    = xl_border()
        c.font      = xl_font(size=9)
        c.alignment = xl_left(wrap=True) if ci == 13 else xl_center()
        if ci == 3:
            c.font = Font(bold=True, size=9, color=_WHITE, name="Calibri")
            c.fill = xl_fill(tc)
        else:
            c.fill = xl_fill(rbg)
    ws2.row_dimensions[ri].height = 16

ws2.freeze_panes = "A3"

# Conditional format on EB Mean column (col 8)
max_r2 = 2 + len(elite_sort)
ws2.conditional_formatting.add(
    f"H3:H{max_r2}",
    ColorScaleRule(start_type="min",  start_color="FFC7CE",
                   mid_type="percentile", mid_value=50, mid_color="FFEB9C",
                   end_type="max",    end_color="C6EFCE")
)
print("Sheet 2: Elite Targets — done")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Sheet 3: Finishing Edge
# ──────────────────────────────────────────────────────────────────────────────
ws3 = wb.create_sheet("Finishing Edge")
ws3.sheet_view.showGridLines = False

xl_title(ws3, 1, 1, 11, "FINISHING EDGE — EB Mean vs xG Rate (sorted by edge, min 10 shots)")

headers3 = ["#", "Player", "Tier", "Shots", "Goals",
            "xG/Shot", "EB Mean", "EB vs xG", "CI Low", "CI High", "CI Width"]
widths3  = [5, 22, 14, 7, 7, 11, 10, 11, 10, 10, 10]
xl_header_row(ws3, 2, headers3, widths3)

for ri, (_, row) in enumerate(finedge_sort.iterrows(), 3):
    tc    = TIER_COLOR_XL.get(row["tier"], "AAAAAA")
    rbg   = _LIGHT_BG if ri % 2 == 1 else _WHITE
    pos   = row["eb_vs_xg"] >= 0
    vals  = [
        ri - 2, row["player_name"], row["tier"],
        int(row["n"]), int(row["goals"]),
        xl_pct(row["xg_per_shot"]), xl_pct(row["eb_mean"]),
        f"{row['eb_vs_xg']*100:+.2f}%",
        xl_pct(row["eb_ci_lo"]),   xl_pct(row["eb_ci_hi"]),
        f"{row['ci_width']*100:.2f}pp",
    ]
    for ci, val in enumerate(vals, 1):
        c = ws3.cell(row=ri, column=ci, value=val)
        c.border    = xl_border()
        c.font      = xl_font(size=9)
        c.alignment = xl_center()
        if ci == 3:
            c.font = Font(bold=True, size=9, color=_WHITE, name="Calibri")
            c.fill = xl_fill(tc)
        elif ci == 8:
            c.fill = xl_fill("C6EFCE" if pos else "FFC7CE")
            c.font = Font(bold=True, size=9,
                         color="006100" if pos else "9C0006", name="Calibri")
        else:
            c.fill = xl_fill(rbg)
    ws3.row_dimensions[ri].height = 16

ws3.freeze_panes = "A3"
print("Sheet 3: Finishing Edge — done")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Sheet 4: Hidden Gems
# ──────────────────────────────────────────────────────────────────────────────
ws4 = wb.create_sheet("Hidden Gems")
ws4.sheet_view.showGridLines = False

xl_title(ws4, 1, 1, 11, "HIDDEN GEMS — Small-Sample High EB Mean (5-29 shots)", bg="F09B2A")

headers4 = ["#", "Player", "Shots", "Goals", "Raw Rate",
            "xG/Shot", "EB Mean", "EB vs xG", "Shrinkage", "Reliability", "Verdict"]
widths4  = [5, 22, 7, 7, 10, 11, 10, 11, 10, 10, 38]
xl_header_row(ws4, 2, headers4, widths4, bg="F09B2A")

gems_export = df[(df['n'] >= 5) & (df['n'] < 30)].sort_values('eb_mean', ascending=False)
gems_export = gems_export.copy()
gems_export['verdict_xl'] = gems_export.apply(hidden_verdict, axis=1)

for ri, (_, row) in enumerate(gems_export.iterrows(), 3):
    rbg  = _LIGHT_BG if ri % 2 == 1 else _WHITE
    vals = [
        ri - 2, row['player_name'], int(row['n']), int(row['goals']),
        xl_pct(row['raw_rate']), xl_pct(row['xg_per_shot']),
        xl_pct(row['eb_mean']),  f"{row['eb_vs_xg']*100:+.2f}%",
        xl_pct(row['shrinkage']), xl_pct(row['reliability']),
        row['verdict_xl'],
    ]
    for ci, val in enumerate(vals, 1):
        c = ws4.cell(row=ri, column=ci, value=val)
        c.border    = xl_border()
        c.font      = xl_font(size=9)
        c.alignment = xl_left() if ci == 11 else xl_center()
        c.fill      = xl_fill(rbg)
    ws4.row_dimensions[ri].height = 16

ws4.freeze_panes = "A3"
print("Sheet 4: Hidden Gems — done")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Sheet 5: Full Rankings
# ──────────────────────────────────────────────────────────────────────────────
ws5 = wb.create_sheet("Full Rankings")
ws5.sheet_view.showGridLines = False

xl_title(ws5, 1, 1, 14, f"FULL PLAYER RANKINGS — All {len(df)} Players, Sorted by EB Mean")

headers5 = ["Rank", "Player", "Tier", "Shots", "Goals", "xG Sum",
            "Raw Conv%", "xG/Shot", "EB Mean", "CI Low", "CI High",
            "EB vs xG", "Shrinkage", "Reliability"]
widths5  = [6, 22, 14, 7, 7, 10, 10, 9, 9, 9, 9, 10, 10, 10]
xl_header_row(ws5, 2, headers5, widths5)

df_ranked = df.sort_values("eb_mean", ascending=False).reset_index(drop=True)

for ri, (_, row) in enumerate(df_ranked.iterrows(), 3):
    tc  = TIER_COLOR_XL.get(row["tier"], "AAAAAA")
    rbg = _LIGHT_BG if ri % 2 == 1 else _WHITE
    vals = [
        ri - 2, row["player_name"], row["tier"],
        int(row["n"]), int(row["goals"]),
        f"{row['xg_sum']:.2f}",
        xl_pct(row["raw_rate"]),     xl_pct(row["xg_per_shot"]),
        xl_pct(row["eb_mean"]),      xl_pct(row["eb_ci_lo"]),
        xl_pct(row["eb_ci_hi"]),
        f"{row['eb_vs_xg']*100:+.2f}%",
        xl_pct(row["shrinkage"]),    xl_pct(row["reliability"]),
    ]
    for ci, val in enumerate(vals, 1):
        c = ws5.cell(row=ri, column=ci, value=val)
        c.border    = xl_border()
        c.font      = xl_font(size=9)
        c.alignment = xl_center()
        if ci == 3:
            c.font = Font(bold=True, size=8, color=_WHITE, name="Calibri")
            c.fill = xl_fill(tc)
        else:
            c.fill = xl_fill(rbg)
    ws5.row_dimensions[ri].height = 15

# Conditional format on EB Mean (col 9) and EB vs xG (col 12)
max_r5 = 2 + len(df_ranked)
ws5.conditional_formatting.add(
    f"I3:I{max_r5}",
    ColorScaleRule(start_type="min", start_color="FFC7CE",
                   mid_type="num",   mid_value=league_mean, mid_color="FFEB9C",
                   end_type="max",   end_color="C6EFCE")
)
ws5.conditional_formatting.add(
    f"L3:L{max_r5}",
    ColorScaleRule(start_type="min", start_color="FFC7CE",
                   mid_type="num",   mid_value=0, mid_color="FFFFFF",
                   end_type="max",   end_color="C6EFCE")
)
ws5.freeze_panes = "A3"

print("Sheet 5: Full Rankings — done")

In [ ]:
# ── Save workbook ─────────────────────────────────────────────────────────────
wb.save(OUTPUT_PATH)
print(f"Excel workbook saved to: {OUTPUT_PATH}")
print(f"Sheets: {[s.title for s in wb.worksheets]}")
print(f"Players in workbook: {len(df)}  |  Qualified (>=10): {len(df_q10)}  |  "
      f"Elite: {(df_q10['tier']=='Elite').sum()}")